# Figure 7 (was S16; notebook ID S01): predicted-structure entropy separates real from spurious small ORFs only weakly (GMSC, 43.7 M smORFs)

**Status:** draft, Gadi. The ID is provisional.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figS01_length_hist.tsv.gz`, `data/figS01_entropy_hist.tsv.gz` | `smorf_entropy.tsv.gz` (913 MB, sha256 `88df8222…`) | `encode_entropy.py`, `merge_plot.py` |
| `data/figS01_single_feature_auc.tsv.gz` | `manuscript/table2_single_feature_auc.tsv` | `manuscript_stats.py` |
| `data/figSS01_density_aa_3di.tsv.gz` (panel e) | `density_aa_vs_3di_counts.npz` (400 × 400 bins of 0.011 bits) | `density_aa_vs_3di.py` (job 181044341) |
| `data/figS01_models_by_split.tsv.gz` | `clusters/cluster_split_models.tsv` | `cluster.pbs` (MMseqs2 14.7e284), `cluster_models.py` (HistGradientBoosting, 2 M train / 2 M test, seed 42) |

- **Run:** `linsalrob/genome_entropy` `Validation/smorfs/` (#105), with the write-up at `agent-handoffs` `projects/genome_entropy_logan/smorfs/manuscript.md`.
  - Software: `genome_entropy` 0.3.0, the ModernProst `gbouras13/modernprost-50M` encoder at revision `2d68d506`, encoding size 50,000, and translation table 11.
  - Hardware: V100 (`gpuvolta`).
- **Results location:** Gadi `/g/data/ob80/re3494/Projects/smORFs/results/`, mirrored on Teams at `genome_entropy:General/smORFs/results/`.
- **Reduction:** `manuscript/prep/smorfs_prepare_data.py`. It produces histograms over the per-smORF table (PBS job 181367025, 0.06 SU) and copies the AUC, model and stripe tables verbatim.
- **Data:** the GMSC benchmark sets as supplied by the GMSC authors.
  - **Positives:** 33,647,794 high-quality smORFs with RNAcode plus transcriptomic or proteomic support, or homology to known small proteins.
  - **Negatives:** 10,079,115 conserved, in-frame GMSC sequences that failed coding tests.
  - Lengths are 10–99 aa, with no 90 aa floor. The negatives are not random ORFs.
- **Binning:** 3Di bins are 0.005 bits wide and closed on the right, so strings at exactly log₂k sit just below the line.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}

import math
from matplotlib.colors import LogNorm, TwoSlopeNorm
CLS = {"positive": "#0072B2", "negative": "#D55E00"}


In [ ]:
L = pd.read_csv(DATA / "figS01_length_hist.tsv.gz", sep="\t")
E = pd.read_csv(DATA / "figS01_entropy_hist.tsv.gz", sep="\t")
A = pd.read_csv(DATA / "figS01_single_feature_auc.tsv.gz", sep="\t")
M = pd.read_csv(DATA / "figS01_models_by_split.tsv.gz", sep="\t")
D = pd.read_csv(DATA / "figSS01_density_aa_3di.tsv.gz", sep="\t")
BINS = ["10-29", "30-49", "50-59", "60-69", "70-79", "80-89", "90-99"]
BLAB = [b.replace("-", "–") for b in BINS]

# Numbers quoted in the caption and the Results (README rule 5).
npos, nneg = (int(L[L.dataset == c].n.sum()) for c in ("positive", "negative"))
assert (npos, nneg) == (33_647_794, 10_079_115)
assert {c: int(D[D.dataset == c].n.sum()) for c in ("positive", "negative")} == {"positive": npos, "negative": nneg}
auc = A.set_index(["length_bin", "feature"]).roc_auc
assert round(auc["all", "aa_length"], 3) == 0.640 and round(auc["all", "three_di_entropy"], 3) == 0.605
assert round(auc["50-59", "three_di_entropy"], 3) == 0.526 and round(auc["90-99", "three_di_entropy"], 3) == 0.626
mod = M.set_index(["split", "model"])
assert [round(mod.roc_auc["random", m], 3) for m in ("length + nt + aa", "length + 3Di + 12-state", "all five")] == [0.669, 0.699, 0.730]
assert [round(mod.roc_auc["cluster_id30_cov80", m], 3) for m in ("length + nt + aa", "length + 3Di + 12-state", "all five")] == [0.662, 0.689, 0.721]
assert round(mod.roc_auc_cluster_weighted["cluster_id30_cov80", "all five"], 3) == 0.695

fig = plt.figure(figsize=(7.2, 7.6))
gs = fig.add_gridspec(3, 3, height_ratios=[1, 1.05, 1.05], hspace=0.62, wspace=0.42)

# (a) length
ax = fig.add_subplot(gs[0, 0])
for c in ("positive", "negative"):
    g = L[L.dataset == c]
    ax.stairs(g.n / g.n.sum() * 100, np.append(g.aa_length - 0.5, g.aa_length.iloc[-1] + 0.5),
              color=CLS[c], lw=0.9, label=f"{c} (n = {g.n.sum() / 1e6:.1f} M)")
ax.set_xlabel("protein length (aa)")
ax.set_ylabel("% of class")
ax.set_ylim(0, 3.6)
ax.legend(frameon=False, fontsize=6, loc="upper left")
panel(ax, "a")

# (b, c) 3Di and amino-acid entropy
for j, (feat, lab, lim, letter) in enumerate((("three_di_entropy", "3Di entropy (bits)", (0, 4.2), "b"),
                                              ("aa_entropy", "amino-acid entropy (bits)", (2.4, 4.4), "c"))):
    ax = fig.add_subplot(gs[0, j + 1])
    for c in ("positive", "negative"):
        g = E[(E.dataset == c) & (E.feature == feat)]
        dens = g.n / (g.n_dataset * (g.bin_hi - g.bin_lo))
        ax.stairs(dens, np.append(g.bin_lo, g.bin_hi.iloc[-1]), color=CLS[c], lw=0.7)
    if feat == "three_di_entropy":
        for v in (1.0, math.log2(3), 2.0):
            ax.axvline(v, color=MUTED, lw=0.5, ls=(0, (1, 2)), zorder=0)
        ax.text(math.log2(3), 0.98, " log$_2$3", transform=ax.get_xaxis_transform(),
                fontsize=5.8, color=MUTED, va="top")
        ax.set_ylim(0, 2.0)
    ax.set_xlim(*lim)
    ax.set_xlabel(lab)
    ax.set_ylabel("density")
    panel(ax, letter)

# (d) single-feature AUC within length bins
ax = fig.add_subplot(gs[1, :2])
FEAT = [("aa_length", "length", "#000000"), ("orf_nt_entropy", "nucleotide entropy", "#999999"),
        ("aa_entropy", "amino-acid entropy", "#E69F00"), ("three_di_entropy", "3Di entropy", "#0072B2"),
        ("twelve_state_entropy", "12-state entropy", "#56B4E9")]
for f, lab, col in FEAT:
    g = A[A.feature == f].set_index("length_bin").reindex(BINS)
    ax.plot(range(len(BINS)), g.roc_auc, marker="o", ms=3, lw=1.0, color=col, label=lab)
ax.axhline(0.5, color=MUTED, lw=0.6, ls="--")
ax.set_xticks(range(len(BINS)), BLAB)
ax.set_xlabel("protein-length bin (aa)")
ax.set_ylabel("single-feature ROC AUC\n(positive vs negative)")
ax.set_ylim(0.34, 0.66)
ax.set_yticks([0.40, 0.45, 0.50, 0.55, 0.60, 0.65])
ax.legend(frameon=False, fontsize=6, ncol=3, loc="lower center")
panel(ax, "d")

# (e) amino-acid x 3Di plane: positive minus negative fraction per bin
edges = np.linspace(0, 4.4, 401)
Hc = {}
for c in ("positive", "negative"):
    g = D[D.dataset == c]
    h = np.zeros((400, 400))
    h[g.i_aa, g.j_3di] = g.n  # h[amino acid, 3Di]
    Hc[c] = h / h.sum()
diff = (Hc["positive"] - Hc["negative"]) * 1e4
lim = np.nanmax(np.abs(diff))
ax = fig.add_subplot(gs[1, 2])
m2 = ax.pcolormesh(edges, edges, np.where((Hc["positive"] + Hc["negative"]) > 0, diff, np.nan).T,
                   norm=TwoSlopeNorm(0, -lim, lim), cmap="RdBu", rasterized=True)
ax.set_xlim(2.4, 4.4)
ax.set_ylim(0, 4.2)
ax.set_xlabel("amino-acid entropy (bits)")
ax.set_ylabel("3Di entropy (bits)")
ax.text(0.03, 0.97, "blue: more positives\nred: more negatives", transform=ax.transAxes,
        ha="left", va="top", fontsize=5.6, color=INK)
cax = ax.inset_axes([1.04, 0.1, 0.05, 0.8])
cb = fig.colorbar(m2, cax=cax)
cb.ax.tick_params(labelsize=5.5)
cb.set_label("positive − negative\nfraction per bin (×10$^{-4}$)", fontsize=5.8)
panel(ax, "e")

# (f) held-out models within length bins
ax = fig.add_subplot(gs[2, :2])
MOD = [("length + nt + aa", "length + nucleotide and amino-acid entropy", "#E69F00"),
       ("length + 3Di + 12-state", "length + 3Di and 12-state entropy", "#0072B2"),
       ("all five", "all five features", "#000000")]
for split, ls in (("random", "-"), ("cluster_id30_cov80", "--")):
    for mname, lab, col in MOD:
        r = mod.loc[(split, mname)]
        ax.plot(range(len(BINS)), [r[f"auc_{b.replace('-', '_')}"] for b in BINS], color=col, ls=ls,
                lw=1.0, marker="o" if ls == "-" else None, ms=3)
ax.axhline(0.5, color=MUTED, lw=0.6, ls=":")
ax.set_xticks(range(len(BINS)), BLAB)
ax.set_xlabel("protein-length bin (aa)")
ax.set_ylabel("held-out ROC AUC")
ax.set_ylim(0.5, 0.76)
panel(ax, "f")
from matplotlib.lines import Line2D
handles = [Line2D([], [], color=col, lw=1.2, label=lab) for _, lab, col in MOD]
handles += [Line2D([], [], color=MUTED, lw=1.0, ls="-", marker="o", ms=3, label="random train/test split"),
            Line2D([], [], color=MUTED, lw=1.0, ls="--", label="test families held out\n(30% identity clusters)")]
lax = fig.add_subplot(gs[2, 2])
lax.axis("off")
lax.legend(handles=handles, frameon=False, fontsize=6, loc="center left", title="models (panel f)",
           title_fontsize=6.5, alignment="left")
save(fig, "figS01_smorf_separation")


In [ ]:
print(A[A.length_bin == "all"][["feature", "roc_auc"]].to_string(index=False))
print(M.pivot_table(index="model", columns="split", values="roc_auc").round(3).to_string())
print(M.pivot_table(index="model", columns="split", values="roc_auc_cluster_weighted").round(3).to_string())


## Caption (draft)

**Figure 7. On curated small ORFs (10–99 aa), predicted-structure entropy separates real from spurious sequences only weakly, but it adds more than composition.**

The data are the GMSC benchmark: 33,647,794 positive and 10,079,115 negative smORFs.
- **(a)** Protein length. Positives are longer (median 80 against 70 aa).
- **(b)** 3Di entropy. Both classes share a mode below log₂3. Positives carry a second mode at 2.5–3.5 bits. Spikes at log₂2 and log₂3 are two- and three-state strings (Figure S19).
- **(c)** Amino-acid entropy. The two classes almost overlap.
- **(d)** Single-feature ROC AUC within length bins. Pooled, length is the best single feature (0.640, against 0.605 for 3Di entropy). At fixed length, amino-acid and nucleotide entropy fall to or below chance (AUC < 0.5 means the feature is higher in negatives). 3Di entropy keeps its signal and it grows with length, from 0.526 at 50–59 aa to 0.626 at 90–99 aa.
- **(e)** Amino-acid against 3Di entropy: the fraction of positives minus the fraction of negatives in each 0.011 × 0.011-bit bin. Positives are enriched above about 2.3 bits of 3Di entropy when amino-acid entropy exceeds about 3.6 bits; negatives in a band at 1.0–1.6 bits of 3Di entropy near 4.0 bits of amino-acid entropy. Thin horizontal lines at 1.0 bit and just below log₂3 are the log₂k stripes (Figure S19). The two classes differ jointly on both axes, so a threshold on either alone cannot separate them. Each class on its own is shown in Figure S18.
- **(f)** Gradient-boosted models within length bins, trained on 2 M sequences and tested on a disjoint 2 M. Solid lines use a random split; dashed lines hold out test families at 30% identity (MMseqs2).
  - Overall AUC: length with nucleotide and amino-acid entropy 0.669; length with 3Di and 12-state entropy 0.699; all five features 0.730.
  - With 30% identity clusters held out these become 0.662, 0.689 and 0.721.
  - With each 50%-identity cluster weighted once, the all-five AUC under the 30% split is 0.695.

For comparison, the same feature families reach AUC ≈ 0.98 on gene-length LOGAN ORFs.
